# HYP Gaussian-Noise Augmentation Ablation (ON vs OFF)

Requested by external review: NB03 applies Gaussian-noise augmentation to
HYP-positive training records (Atwa et al. 2025's recipe) and presents it as
a fix for HYP's weak per-class AUC, but never ablates it — and Atwa et al.
2025 themselves note this kind of noise augmentation can hurt. This notebook
runs the identical InceptionTime1D training pipeline from NB03 with the
augmentation ON (reusing NB03's already-trained checkpoint where possible)
and OFF (fresh training run on the un-augmented training set), then reports
macro-AUC, per-class AUC/AUPRC/F1 (especially HYP), and Fmax for both, on
the untouched TEST split.

Same seed (42), same architecture, same 10-epoch/EarlyStopping/
ReduceLROnPlateau budget as NB03 Module 6-7 — augmentation is the only
variable that changes between the two conditions.

## Setup

In [1]:
import os, sys, time, random, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, hamming_loss, average_precision_score
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [2]:
PROJECT_ROOT   = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR    = PROJECT_ROOT / 'outputs'
OUTPUTS_VAL    = OUTPUTS_DIR / 'dataset_validation'
OUTPUTS_VAL.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SUPERCLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LABEL_COLS   = ['is_NORM', 'is_MI', 'is_STTC', 'is_CD', 'is_HYP']
NUM_CLASSES  = len(SUPERCLASSES)
AUG_NOISE_STD = 0.01  # Atwa et al. 2025, Table 4 -- identical to NB03

MAX_EPOCHS         = 10
EARLY_STOP_PATIENCE = 4
LR_PLATEAU_PATIENCE = 2

ON_CKPT  = OUTPUTS_DIR / '05_optimized_InceptionTime1D_best.pth'   # NB03's existing ON-augmentation checkpoint
OFF_CKPT = OUTPUTS_DIR / '03b_ablation_noaug_InceptionTime1D_best.pth'
RESULTS_JSON = OUTPUTS_VAL / '03b_hyp_augmentation_ablation.json'

DEVICE

device(type='cpu')

## Data loading and augmentation

In [3]:
def load_data():
    meta = pd.read_csv(DATA_PROCESSED / 'MASTER_RESEARCH_METADATA.csv')
    df_train = meta[meta['split'] == 'TRAIN'].reset_index(drop=True)
    df_val   = meta[meta['split'] == 'VALIDATION'].reset_index(drop=True)
    df_test  = meta[meta['split'] == 'TEST'].reset_index(drop=True)

    X_train = np.load(DATA_PROCESSED / 'X_train_clean_100.npy')
    X_val   = np.load(DATA_PROCESSED / 'X_val_clean_100.npy')
    X_test  = np.load(DATA_PROCESSED / 'X_test_clean_100.npy')

    y_train = df_train[LABEL_COLS].values.astype(np.float32)
    y_val   = df_val[LABEL_COLS].values.astype(np.float32)
    y_test  = df_test[LABEL_COLS].values.astype(np.float32)
    return X_train, y_train, X_val, y_val, X_test, y_test


def augment_hyp(X_train, y_train):
    rng = np.random.default_rng(SEED)
    hyp_mask = y_train[:, SUPERCLASSES.index('HYP')] == 1
    X_hyp_aug = X_train[hyp_mask] + rng.normal(0.0, AUG_NOISE_STD, X_train[hyp_mask].shape).astype(np.float32)
    y_hyp_aug = y_train[hyp_mask].copy()
    return np.concatenate([X_train, X_hyp_aug], axis=0), np.concatenate([y_train, y_hyp_aug], axis=0)


class ECGDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals = torch.tensor(signals, dtype=torch.float32).transpose(1, 2)
        self.labels  = torch.tensor(labels,  dtype=torch.float32)
    def __len__(self): return len(self.signals)
    def __getitem__(self, idx): return self.signals[idx], self.labels[idx]

## Model: InceptionTime1D

In [4]:
class InceptionBlock1D(nn.Module):
    def __init__(self, in_channels, nb_filters=32, kernel_sizes=(9, 19, 39)):
        super().__init__()
        self.bottleneck = nn.Conv1d(in_channels, nb_filters, 1, bias=False)
        self.convs = nn.ModuleList([
            nn.Conv1d(nb_filters, nb_filters, k, padding=k // 2, bias=False)
            for k in kernel_sizes
        ])
        self.pool_conv = nn.Conv1d(in_channels, nb_filters, 1, bias=False)
        self.pool      = nn.MaxPool1d(3, stride=1, padding=1)
        self.bn  = nn.BatchNorm1d(nb_filters * (len(kernel_sizes) + 1))
        self.act = nn.ReLU()

    def forward(self, x):
        bottleneck_out = self.bottleneck(x)
        conv_outs      = [conv(bottleneck_out) for conv in self.convs]
        pool_out       = self.pool_conv(self.pool(x))
        return self.act(self.bn(torch.cat(conv_outs + [pool_out], dim=1)))


class InceptionTime1D(nn.Module):
    def __init__(self, num_classes=5, in_channels=12, nb_filters=32, depth=4):
        super().__init__()
        channels = [in_channels] + [nb_filters * 4] * depth
        self.blocks = nn.Sequential(*[
            InceptionBlock1D(channels[i], nb_filters) for i in range(depth)
        ])
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Linear(nb_filters * 4, num_classes)
        self.drop = nn.Dropout(0.3)

    def forward(self, x):
        x = self.blocks(x)
        x = self.gap(x).squeeze(-1)
        return self.fc(self.drop(x))

## Training and evaluation utilities

In [5]:
def macro_auc(Y, P):
    scores = [roc_auc_score(Y[:, k], P[:, k]) for k in range(Y.shape[1]) if len(np.unique(Y[:, k])) == 2]
    return float(np.mean(scores)) if scores else float('nan')


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for sig, targets in loader:
        sig, targets = sig.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(sig), targets)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(targets)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    for sig, targets in loader:
        probs = torch.sigmoid(model(sig.to(DEVICE))).cpu().numpy()
        all_preds.append(probs); all_targets.append(targets.numpy())
    return np.vstack(all_preds), np.vstack(all_targets)


def train_inceptiontime(X_train_cond, y_train_cond, X_val, y_val, ckpt_path, tag):
    train_ds = ECGDataset(X_train_cond, y_train_cond)
    val_ds   = ECGDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)

    pos_counts = y_train_cond.sum(axis=0)
    neg_counts = len(y_train_cond) - pos_counts
    pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    torch.manual_seed(SEED)  # re-seed so both conditions start from the same init
    model = InceptionTime1D(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=LR_PLATEAU_PATIENCE)

    if ckpt_path.exists():
        print(f'[{tag}] Found existing checkpoint {ckpt_path.name} -> reusing (no retraining).')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        return model

    best_auc, epochs_no_improve = 0.0, 0
    print(f'[{tag}] Training InceptionTime1D for up to {MAX_EPOCHS} epochs on {DEVICE} '
          f'(train N={len(X_train_cond)})...')
    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_preds, val_tgts = predict(model, val_loader)
        val_auc = macro_auc(val_tgts, val_preds)
        scheduler.step(val_auc)
        print(f'  [{tag}] Epoch {epoch:02d}/{MAX_EPOCHS} | Loss: {tr_loss:.4f} | Val AUC: {val_auc:.4f}')
        if val_auc > best_auc:
            best_auc, epochs_no_improve = val_auc, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f'  [{tag}] EarlyStopping at epoch {epoch}.')
                break
    print(f'[{tag}] Done in {time.time()-t0:.1f}s. Best Val AUC: {best_auc:.4f}')
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    return model


def fmax(y_true, y_prob, taus=np.arange(0.05, 0.96, 0.01)):
    best = -1.0
    for t in taus:
        pred = (y_prob >= t).astype(int)
        f = np.mean([f1_score(y_true[:, k], pred[:, k], zero_division=0) for k in range(y_true.shape[1])])
        best = max(best, f)
    return best


def macro_auprc(y_true, y_prob):
    return float(np.mean([average_precision_score(y_true[:, k], y_prob[:, k]) for k in range(y_true.shape[1])]))


def per_class_metrics(y_true, y_prob):
    out = {}
    for i, cls in enumerate(SUPERCLASSES):
        auc = roc_auc_score(y_true[:, i], y_prob[:, i]) if len(np.unique(y_true[:, i])) == 2 else float('nan')
        auprc = average_precision_score(y_true[:, i], y_prob[:, i])
        out[cls] = {'auc': float(auc), 'auprc': float(auprc)}
    return out


def evaluate(model, X_test, y_test, tag):
    test_ds = ECGDataset(X_test, y_test)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)
    test_preds, test_tgts = predict(model, test_loader)
    result = {
        'macro_auc': macro_auc(test_tgts, test_preds),
        'macro_auprc': macro_auprc(test_tgts, test_preds),
        'fmax': fmax(test_tgts, test_preds),
        'per_class': per_class_metrics(test_tgts, test_preds),
    }
    print(f'[{tag}] TEST macro-AUC={result["macro_auc"]:.4f}  macro-AUPRC={result["macro_auprc"]:.4f}  '
          f'Fmax={result["fmax"]:.4f}  HYP-AUC={result["per_class"]["HYP"]["auc"]:.4f}  '
          f'HYP-AUPRC={result["per_class"]["HYP"]["auprc"]:.4f}')
    return result

## Load data

In [6]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'[DATA] Train N={len(X_train)}  Val N={len(X_val)}  Test N={len(X_test)}')

[DATA] Train N=17418  Val N=1080  Test N=2198


## Condition ON: NB03's recipe (Gaussian noise added to HYP-positive records)

In [7]:
X_train_on, y_train_on = augment_hyp(X_train, y_train)
print(f'[AUG-ON] Train N: {len(X_train)} -> {len(X_train_on)}')
model_on = train_inceptiontime(X_train_on, y_train_on, X_val, y_val, ON_CKPT, 'AUG-ON')
result_on = evaluate(model_on, X_test, y_test, 'AUG-ON')

[AUG-ON] Train N: 17418 -> 19537
[AUG-ON] Found existing checkpoint 05_optimized_InceptionTime1D_best.pth -> reusing (no retraining).
[AUG-ON] TEST macro-AUC=0.9226  macro-AUPRC=0.8083  Fmax=0.7411  HYP-AUC=0.8955  HYP-AUPRC=0.6530


## Condition OFF: identical pipeline, no augmentation

In [8]:
model_off = train_inceptiontime(X_train, y_train, X_val, y_val, OFF_CKPT, 'AUG-OFF')
result_off = evaluate(model_off, X_test, y_test, 'AUG-OFF')

[AUG-OFF] Found existing checkpoint 03b_ablation_noaug_InceptionTime1D_best.pth -> reusing (no retraining).
[AUG-OFF] TEST macro-AUC=0.9232  macro-AUPRC=0.8085  Fmax=0.7255  HYP-AUC=0.9027  HYP-AUPRC=0.6608


## Summary

In [9]:
print('='*100)
print('SUMMARY')
print('='*100)
print(f'{"Metric":<16s} {"AUG-ON":>10s} {"AUG-OFF":>10s} {"Delta (ON-OFF)":>16s}')
for k in ['macro_auc', 'macro_auprc', 'fmax']:
    print(f'{k:<16s} {result_on[k]:>10.4f} {result_off[k]:>10.4f} {result_on[k]-result_off[k]:>16.4f}')
for cls in SUPERCLASSES:
    auc_on, auc_off = result_on['per_class'][cls]['auc'], result_off['per_class'][cls]['auc']
    auprc_on, auprc_off = result_on['per_class'][cls]['auprc'], result_off['per_class'][cls]['auprc']
    print(f'{cls+"-AUC":<16s} {auc_on:>10.4f} {auc_off:>10.4f} {auc_on-auc_off:>16.4f}')
    print(f'{cls+"-AUPRC":<16s} {auprc_on:>10.4f} {auprc_off:>10.4f} {auprc_on-auprc_off:>16.4f}')

with open(RESULTS_JSON, 'w') as f:
    json.dump({'aug_on': result_on, 'aug_off': result_off}, f, indent=2)
print(f'\n[SAVED] {RESULTS_JSON}')

SUMMARY
Metric               AUG-ON    AUG-OFF   Delta (ON-OFF)
macro_auc            0.9226     0.9232          -0.0006
macro_auprc          0.8083     0.8085          -0.0002
fmax                 0.7411     0.7255           0.0157
NORM-AUC             0.9456     0.9458          -0.0003
NORM-AUPRC           0.9225     0.9217           0.0008
MI-AUC               0.9247     0.9201           0.0046
MI-AUPRC             0.8330     0.8273           0.0056
STTC-AUC             0.9340     0.9285           0.0055
STTC-AUPRC           0.8226     0.8088           0.0138
CD-AUC               0.9132     0.9188          -0.0057
CD-AUPRC             0.8103     0.8239          -0.0136
HYP-AUC              0.8955     0.9027          -0.0073
HYP-AUPRC            0.6530     0.6608          -0.0078

[SAVED] /home/tandon/Teep/age with ecg/outputs/dataset_validation/03b_hyp_augmentation_ablation.json
